# 16.3 生成時重算了什麼？


模型讀完 `[1,2,3]`，再加入ID4，最直接的做法是把 `[1,2,3,4]`整段重新送入。可是前面三個位置的Key、Value不是已經算過了嗎？因果注意力只允許每個位置讀到自己和過去，因此加入未來的ID4不會改變前面三個位置的特徵；我們可以保留它們各層的K、V，下次只算新位置。本節要先核對這樣重用是否保持結果相同，再談省時間。

沿用[prefill/decode](https://birdhackor.github.io/tiny-perceptron-vlm/16.2.html)的分工，這次把例子縮成三個前文位置，再接第四個ID。cache保存每層注意力裡的Key與Value，沒有把完整回答存成答案，也沒有免去新位置的所有工作。新token仍要查表、算Query與自己的K/V、對所有過去位置做注意力，再經過[FFN前饋網路](https://birdhackor.github.io/tiny-perceptron-vlm/4.4.html)，也就是對每個位置各自轉換特徵的兩層網路；省掉的是舊位置重複的投影與特徵處理。

圖中的0、1、2、3是位置編號，並非本例輸入ID。最上列的K0/V0至K2/V2是已保存的三個位置；中間算出新位置3的K/V，再接到快取尾端，成為四格。最下面的新Query讀這四格，不能只比較剛加入的一格。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/rewrite-16-cache-append.svg")))

In [ ]:
import torch
from tiny_perceptron.model import TinyLM, ModelConfig

torch.manual_seed(0)
model = TinyLM(ModelConfig(width=8)).eval()
ids = torch.tensor([[1, 2, 3, 4]])
with torch.no_grad():
    full = model(ids)["logits"][:, -1]
    prefix = model(ids[:, :3])
    step = model(ids[:, 3:], cache=prefix["cache"])
    cached = step["logits"][:, 0]
error = (full - cached).abs().max().item()
print("兩路分數形狀", tuple(full.shape), tuple(cached.shape))
print("最大差異", error)
assert torch.allclose(full, cached, atol=1e-6)

`ids[:,:3]`取前三個ID，`ids[:,3:]`只取第四個。重算整段時最後位置索引為-1；快取路徑只有新位置，所以取索引0。兩者都是用讀完ID4後的狀態預測下一個token，不能拿prefill最後位置的分數與它比較，因為那是只讀到ID3時的預測。兩路形狀應同為 `(1,264)`，最大差異通常為零或很小的浮點尾數，assert要求接近到指定容忍範圍。

位置也必須對齊。新位置不能重新當成位置0，程式根據cache已有三格，自動從位置3開始；若忘記這個offset（起始偏移），即使K/V形狀正常，結果也可能不同。若改了前文、權重或位置規則，舊cache通常要重建，不能繼續沿用。

多段輸入放在同一批一起處理叫batch。若一段ID為 `[1,2,3]`，另一段為 `[4,5]`，為湊成同長度的表，第二段可補成 `[4,5,PAD]`；padding是補齊的動作，PAD是專用佔位ID，第二段的有效長度仍是2，補格不是真實前文。此小實作沒有處理同batch內這種不同有效長度的快取，所以本節固定長度例子不能直接保證這種輸入也正確。

練習只把最後ID4改成ID5，仍用相同前三個ID的prefix cache；先預測兩路仍接近，再執行核對。接著把第一個ID改掉，必須重新建立prefix。區分這兩種改動，你就知道快取保存的是哪一段計算，而非一份對所有問題都通用的答案。

<details>
<summary>選讀：來源、量測條件與原始實報</summary>

我們也逐步核對了[16.2](https://birdhackor.github.io/tiny-perceptron-vlm/16.2.html)的25位置提示與12步生成。兩個模型都有四個Query頭：MHA各頭使用自己的一組K/V，另一個模型則讓四個Query頭共用一組K/V，所以稱「一個KV頭」，設定詳見[16.4](https://birdhackor.github.io/tiny-perceptron-vlm/16.4.html)。每個模型各自比較整段重算與快取，兩組的最高logit絕對差都為1.90735×10⁻⁶，12個最高分生成ID逐一相同。以絕對容差10⁻⁵核對這次FP32（每個數用32位元表示的浮點數格式）路徑，差異在範圍內；它不是每個數字逐bit相等，也不是任意提示、精度與長度的保證。模型之間的權重不同，這裡只比較每個模型自己的兩條路。

原始ID中多次出現EOS，是速度測試刻意不提早停止的結果；解碼後的 `eueeuare` 或 `eredquare` 會隱去這些結構ID。故報告另保留`generated_ids_full`與`generated_ids_cached`，不能只看解碼文字一樣就宣布序列相同。若兩個最高分候選非常接近，微小浮點差也可能改變選擇，因此還要同時看分數誤差與實際ID。

</details>
